In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from scipy.stats import pearsonr
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === CONFIGURATION ===
proportions_file = 'TOO-Decon/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon'
results_dir = os.path.join(root_dir, 'Results-Pearson-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

# Use this method list for detection
method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)
prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)
prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})
prop_df = prop_df.drop(columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'], errors='ignore')

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# Add missing tissue columns and normalize
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df_tissues = prop_df[reference_tissues].copy()
prop_df_tissues = prop_df_tissues.div(prop_df_tissues.sum(axis=1), axis=0).multiply(100).round(5)
prop_df.update(prop_df_tissues)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_TOOv2_1000', '*_modified.txt')))

# Collect pearson results for tissues
results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))
        matrix = dir_name.replace('Decon-Results_', '').split('-')[0]

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        plot_name_base = f'{matrix}_Random-TOO-V2_{method}'
        plot_pdf = os.path.join(results_dir, plot_name_base + ".pdf")
        plot_svg = os.path.join(results_dir, plot_name_base + ".svg")

        # Load decon data
        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)
        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0
        decon_df = decon_df[reference_tissues]

        # Align samples
        common_samples = prop_df.index.intersection(decon_df.index)
        true_vals_df = prop_df.loc[common_samples, reference_tissues]
        pred_vals_df = decon_df.loc[common_samples, reference_tissues]

        # Flatten and remove zero-zero pairs
        true_vals_flat = true_vals_df.values.flatten()
        pred_vals_flat = pred_vals_df.values.flatten()
        mask_nonzero = ~((true_vals_flat == 0) & (pred_vals_flat == 0))
        true_vals_flat = true_vals_flat[mask_nonzero]
        pred_vals_flat = pred_vals_flat[mask_nonzero]

        # Pearson correlation
        r, p = pearsonr(true_vals_flat, pred_vals_flat)
        print(f"{plot_name_base} — Pearson r = {r:.4f}, p = {p:.2e}")
        results.append([matrix, method, r, r**2, p])
        
        # Plot
        plt.figure(figsize=(3, 3))
        sns.set_theme(style="white")
        sns.set_context("paper", font_scale=1.0)
        ax = sns.regplot(
            x=true_vals_flat, 
            y=pred_vals_flat, 
            scatter_kws={'s': 10, 'alpha': 0.4, 'color': 'black','facecolors': 'none'},
            line_kws={'color': 'crimson', 'lw': 2}
        )
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_xlabel('Actual Fraction (%)', fontsize=10)
        ax.set_ylabel('Predicted Fraction (%)', fontsize=10)
        ax.set_title(f'{matrix} — {method}\nPearson r = {r:.2f}, p = {p:.2e}', fontsize=10)
        ax.tick_params(axis='both', labelsize=7.5, width=0.8, length=4)
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)
        plt.tight_layout()
        plt.savefig(plot_pdf, bbox_inches="tight")
        plt.savefig(plot_svg)
        plt.close()
        print(f"Saved plot: {plot_pdf} and {plot_svg}")

        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Step 2: Save summary table
if results:
    results_df = pd.DataFrame(results, columns=["Matrix", "Method", "Pearson_r", "R_squared", "p_value"])
    results_csv = os.path.join(results_dir, "Random_Pearson_Summary.csv")
    results_df.to_csv(results_csv, index=False)
    print(f"Saved Pearson summary table: {results_csv}")
else:
    print("No results to save.")


2Median_1000_1500_Random-TOO-V2_ReDeconv — Pearson r = -0.1410, p = 5.25e-53
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_ReDeconv.pdf and TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_ReDeconv.svg
2Median_1000_1500_Random-TOO-V2_BayesPrism — Pearson r = 0.6076, p = 0.00e+00
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_BayesPrism.pdf and TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_BayesPrism.svg
2Median_1000_1500_Random-TOO-V2_MuSiC — Pearson r = 0.2001, p = 3.19e-158
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_MuSiC.pdf and TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_MuSiC.svg
2Median_1000_1500_Random-TOO-V2_CIBERSORTx — Pearson r = 0.4247, p = 0.00e+00
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Random-TOO-V2_CIBERSORTx.pdf and TOO-De

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
proportions_file = 'TOO-Decon/20250616_All-Tissues-NoDup_Uniform_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon'
results_dir = os.path.join(root_dir, 'Results-Pearson-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

# Use this method list for detection
method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)
prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)
prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})
prop_df = prop_df.drop(columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'], errors='ignore')

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# Add missing tissue columns and normalize
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df_tissues = prop_df[reference_tissues].copy()
prop_df_tissues = prop_df_tissues.div(prop_df_tissues.sum(axis=1), axis=0).multiply(100).round(5)
prop_df.update(prop_df_tissues)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Uniform_TOOv2_250', '*_modified.txt')))

# Collect pearson results for tissues
results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))
        matrix = dir_name.replace('Decon-Results_', '').split('-')[0]

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        plot_name_base = f'{matrix}_Uniform-TOO-V2_{method}'
        plot_png = os.path.join(results_dir, plot_name_base + ".png")
        plot_pdf = os.path.join(results_dir, plot_name_base + ".pdf")

        # Load decon data
        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)
        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0
        decon_df = decon_df[reference_tissues]

        # Align samples
        common_samples = prop_df.index.intersection(decon_df.index)
        true_vals_df = prop_df.loc[common_samples, reference_tissues]
        pred_vals_df = decon_df.loc[common_samples, reference_tissues]

        # Flatten and remove zero-zero pairs
        true_vals_flat = true_vals_df.values.flatten()
        pred_vals_flat = pred_vals_df.values.flatten()
        mask_nonzero = ~((true_vals_flat == 0) & (pred_vals_flat == 0))
        true_vals_flat = true_vals_flat[mask_nonzero]
        pred_vals_flat = pred_vals_flat[mask_nonzero]

        # Pearson correlation
        r, p = pearsonr(true_vals_flat, pred_vals_flat)
        print(f"{plot_name_base} — Pearson r = {r:.4f}, p = {p:.2e}")
        results.append([matrix, method, r, r**2, p])
        
        # Plot
        plt.figure(figsize=(8, 8))
        sns.set(style="white", context="talk")
        ax = sns.regplot(
            x=true_vals_flat, 
            y=pred_vals_flat, 
            scatter_kws={'s': 30, 'alpha': 0.5, 'color': '#4C72B0'},
            line_kws={'color': 'crimson', 'lw': 2}
        )
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_xlabel('Actual Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Predicted Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_title(f'{matrix} — {method}\nPearson r = {r:.2f}, p = {p:.2e}', fontsize=16, fontweight='bold')
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)
        plt.tight_layout()
        plt.savefig(plot_png, dpi=300, bbox_inches="tight")
        plt.savefig(plot_pdf, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved plot: {plot_png} and {plot_pdf}")

        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Step 2: Save summary table
if results:
    results_df = pd.DataFrame(results, columns=["Matrix", "Method", "Pearson_r", "R_squared", "p_value"])
    results_csv = os.path.join(results_dir, "Uniform_Pearson_Summary.csv")
    results_df.to_csv(results_csv, index=False)
    print(f"Saved Pearson summary table: {results_csv}")
else:
    print("No results to save.")

2Median_1000_1500_Uniform-TOO-V2_ReDeconv — Pearson r = -0.2112, p = 8.87e-30
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_ReDeconv.png and TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_ReDeconv.pdf
2Median_1000_1500_Uniform-TOO-V2_BayesPrism — Pearson r = 0.5234, p = 0.00e+00
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_BayesPrism.png and TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_BayesPrism.pdf
2Median_1000_1500_Uniform-TOO-V2_MuSiC — Pearson r = 0.1246, p = 2.08e-16
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_MuSiC.png and TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_MuSiC.pdf
2Median_1000_1500_Uniform-TOO-V2_CIBERSORTx — Pearson r = 0.3108, p = 2.11e-99
Saved plot: TOO-Decon/Results-Pearson-PredictedVSActual/2Median_1000_1500_Uniform-TOO-V2_CIBERSORTx.png 